# Prototype 6: V1 Paper Trading Bot

The strongest strategy identified was a **one-session mean-reversion** trade:

- Identify stocks that fall by at least 8% from one daily close to the next.
- Enter a long position during the following trading session.
- Exit the position during that same session.

The V1 paper-trading system aims to:

1. Monitor the existing 10-stock universe.
2. Detect qualifying large-drop signals from completed daily market data.
3. Connect to a simulated brokerage account.
4. Submit paper trades when a valid signal occurs.
5. Exit those trades within the same trading session.
6. Record each trade for future out-of-sample evaluation.

## 1. Configuration

For V1, the strategy rules remain intentionally simple and consistent with historical experiments:

- Universe: 10 large-cap US stocks used throughout the project
- Signal threshold: a daily close-to-close return of -8% or lower
- Direction: long only
- Holding period: one trading session
- Environment: paper trading only

Position sizing and execution settings are also defined here so they can be changed later without rewriting the trading logic.

In [1]:
# Stock universe used throughout Prototypes 1–5
TICKERS = [
    "AAPL",
    "MSFT",
    "AMZN",
    "GOOGL",
    "META",
    "JPM",
    "JNJ",
    "XOM",
    "WMT",
    "NVDA"
]

# Strategy rule:
# Generate a signal when the most recent completed daily return is less than or equal to -8%.
DROP_THRESHOLD = -0.08

# Portfolio sizing (how much capital to allocate to a single asset)
POSITION_SIZE_PCT = 0.05

# V1 is paper trading only.
PAPER_TRADING = True

# Name used when labelling orders and saved trade records.
STRATEGY_NAME = "large_drop_v1"

# Exit V1 positions this many minutes before the regular market close.
EXIT_MINUTES_BEFORE_CLOSE = 10

print("V1 configuration loaded.")
print(f"Universe size: {len(TICKERS)} stocks")
print(f"Drop threshold: {DROP_THRESHOLD:.0%}")
print(f"Position size per signal: {POSITION_SIZE_PCT:.0%} of account equity")
print(f"Paper trading: {PAPER_TRADING}")

V1 configuration loaded.
Universe size: 10 stocks
Drop threshold: -8%
Position size per signal: 5% of account equity
Paper trading: True


## 2. Connecting to the Paper Broker

The V1 system uses Alpaca's paper-trading environment to simulate order execution.

1. Load the Alpaca Python SDK.
2. Read the paper-account API credentials.
3. Create a paper-trading client.
4. Confirm that the connection works by retrieving basic account information.

In [2]:
%pip install alpaca-py -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from getpass import getpass

if not os.getenv("ALPACA_API_KEY"):
    os.environ["ALPACA_API_KEY"] = getpass(
        "Enter your Alpaca Paper API Key: "
    )

if not os.getenv("ALPACA_SECRET_KEY"):
    os.environ["ALPACA_SECRET_KEY"] = getpass(
        "Enter your Alpaca Paper Secret Key: "
    )

print("Paper-trading credentials loaded.")

Enter your Alpaca Paper API Key:  ········
Enter your Alpaca Paper Secret Key:  ········


Paper-trading credentials loaded.


In [4]:
from alpaca.trading.client import TradingClient

# Retrieve credentials from environment variables
API_KEY = os.getenv("ALPACA_API_KEY")
SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")

# Create a connection to Alpaca's PAPER trading environment
trading_client = TradingClient(
    API_KEY,
    SECRET_KEY,
    paper=PAPER_TRADING #True
)

print("Trading client created.")
print(f"Paper mode: {PAPER_TRADING}")

Trading client created.
Paper mode: True


In [5]:
# Retrieve basic information from the paper-trading account
account = trading_client.get_account()

print("Connection successful.")
print()
print(f"Account status: {account.status}")
print(f"Portfolio equity: ${float(account.equity):,.2f}")
print(f"Cash: ${float(account.cash):,.2f}")
print(f"Buying power: ${float(account.buying_power):,.2f}")
print(f"Trading blocked: {account.trading_blocked}")

#safety check
if account.trading_blocked:
    raise RuntimeError(
        "Paper account is currently blocked from trading."
    )

print("Paper account is available for trading.")

Connection successful.

Account status: AccountStatus.ACTIVE
Portfolio equity: $100,000.00
Cash: $100,000.00
Buying power: $400,000.00
Trading blocked: False
Paper account is available for trading.


## 3. Checking the Market

Before the bot performs any trading logic, it needs to understand the current market session.

A live trading system cannot assume that every calendar day is a trading day or that the market is always open. Weekends, market holidays, early-closing sessions, and the current time all affect whether an order should be submitted.

Alpaca provides a market clock that reports:

- The current market timestamp.
- Whether the US equity market is currently open.
- The next scheduled market open.
- The next scheduled market close.

This allows the bot to base its behaviour on the actual trading calendar.

In [6]:
# Retrieve the current US market clock

clock = trading_client.get_clock()

print("Market clock retrieved.")
print()
print(f"Current market time: {clock.timestamp}")

Market clock retrieved.

Current market time: 2026-09-18 13:17:03.286070-04:00


In [7]:
# Display a simple interpretation of the current market state

if clock.is_open:
    print("The US stock market is currently OPEN.")
    print(f"Today's market close: {clock.next_close}")
else:
    print("The US stock market is currently CLOSED.")
    print(f"Next market open: {clock.next_open}")

# Store the current market state for later sections
MARKET_IS_OPEN = clock.is_open
NEXT_MARKET_OPEN = clock.next_open
NEXT_MARKET_CLOSE = clock.next_close

The US stock market is currently OPEN.
Today's market close: 2026-09-18 16:00:00-04:00


## 4. Retrieving Recent Market Data

The trading signal is based on the percentage change between **two consecutive daily closing prices**.

This section retrieves recent daily **OHLCV** bars using Alpaca's historical market-data API.

For each trading day, a bar contains:

- **O**pen price
- **H**igh price
- **L**ow price
- **C**lose price
- **T**rading volume

Although the V1 signal only requires **closing prices**, retaining the complete daily bars makes the retrieved data easier to inspect and leaves a clear record of the underlying market information.

In [8]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

# Trading Client: Account, Orders, Positions, Market Clock 
# Data Client: Prices, Daily Bars, Historical US Stock Market Data

data_client = StockHistoricalDataClient(
    API_KEY,
    SECRET_KEY
)

print("Market data client created.")

Market data client created.


In [9]:
from datetime import timedelta

# Retrieve enough recent history to safely contain
# several completed trading sessions.
LOOKBACK_DAYS = 10

data_end = clock.timestamp
data_start = data_end - timedelta(days=LOOKBACK_DAYS)

print(f"Data start: {data_start}")
print(f"Data end:   {data_end}")

Data start: 2026-09-08 13:17:03.286070-04:00
Data end:   2026-09-18 13:17:03.286070-04:00


IEX represents only part of total US trading activity, so its OHLC bars can differ slightly from the full-market SIP bars. If greater pricing precision is required, the Alpaca plan will need to be upgraded to access SIP.

In [10]:
from alpaca.data.enums import DataFeed
# Define the historical-data request
# Use the IEX feed because it is available on Alpaca's free/basic plan.

bars_request = StockBarsRequest(
    symbol_or_symbols=TICKERS, #10 large-cap entities
    timeframe=TimeFrame.Day, #looking at per day granularity
    start=data_start, #time range
    end=data_end, #time range
    feed=DataFeed.IEX
)

# Retrieve the market data
bars = data_client.get_stock_bars(bars_request)

print("Recent daily market data retrieved using the IEX feed.")

Recent daily market data retrieved using the IEX feed.


In [11]:
# Convert the returned bars into a pandas DataFrame

market_data = bars.df.copy()

market_data.head(10)

open     high      low    close  \
symbol timestamp                                                       
AAPL   2026-09-09 04:00:00+00:00  315.590  319.120  309.920  315.420   
       2026-09-10 04:00:00+00:00  316.760  326.660  316.635  326.615   
       2026-09-11 04:00:00+00:00  327.450  336.210  326.570  332.245   
       2026-09-14 04:00:00+00:00  334.700  335.500  331.350  333.000   
       2026-09-15 04:00:00+00:00  330.445  331.780  328.410  331.335   
       2026-09-16 04:00:00+00:00  332.440  335.470  330.710  332.490   
       2026-09-17 04:00:00+00:00  334.805  338.310  330.230  337.090   
       2026-09-18 04:00:00+00:00  337.960  338.410  332.545  334.775   
GOOGL  2026-09-09 04:00:00+00:00  331.530  331.685  327.920  330.655   
       2026-09-10 04:00:00+00:00  328.165  333.210  327.790  332.670   

                                     volume  trade_count        vwap  
symbol timestamp                                                      
AAPL   2026-09-09 04:00:00+00:00  2458568.0      40824.0  313.953332  
       2026-09-10 04:00:00+00:00  2202355.0      41969.0  323.072279  
       2026-09-11 04:00:00+00:00  1563982.0      30142.0  333.431583  
       2026-09-14 04:00:00+00:00  1335636.0      26530.0  333.718123  
       2026-09-15 04:00:00+00:00   930184.0      23920.0  330.277376  
       2026-09-16 04:00:00+00:00   903206.0      17726.0  333.187720  
       2026-09-17 04:00:00+00:00   894177.0      21262.0  334.916974  
       2026-09-18 04:00:00+00:00   692482.0      14184.0  334.847499  
GOOGL  2026-09-09 04:00:00+00:00  1192931.0      26051.0  329.900666  
       2026-09-10 04:00:00+00:00   475560.0      12777.0  330.940105

In [12]:
# Inspect the most recent daily bars for one stock

market_data.loc["AAPL"].tail()

,open,high,low,close,volume,trade_count,vwap
timestamp,,,,,,,
2026-09-14 04:00:00+00:00,334.700,335.50,331.350,333.000,1335636.0,26530.0,333.718123
2026-09-15 04:00:00+00:00,330.445,331.78,328.410,331.335,930184.0,23920.0,330.277376
2026-09-16 04:00:00+00:00,332.440,335.47,330.710,332.490,903206.0,17726.0,333.187720
2026-09-17 04:00:00+00:00,334.805,338.31,330.230,337.090,894177.0,21262.0,334.916974
2026-09-18 04:00:00+00:00,337.960,338.41,332.545,334.775,692482.0,14184.0,334.847499


In [13]:
# Check that market data was returned for every stock
# in the configured V1 universe.

returned_tickers = set(
    market_data.index.get_level_values("symbol").unique()
)

missing_tickers = set(TICKERS) - returned_tickers

if missing_tickers:
    print(f"WARNING: Missing market data for: {sorted(missing_tickers)}")
else:
    print("Market data successfully retrieved for all V1 tickers.")

Market data successfully retrieved for all V1 tickers.


In [14]:
# Display the two most recent available daily bars
# for each stock in the V1 universe.

recent_bars = (
    market_data
    .groupby(level="symbol")
    .tail(2)
)

recent_bars

open     high      low    close  \
symbol timestamp                                                       
AAPL   2026-09-17 04:00:00+00:00  334.805  338.310  330.230  337.090   
       2026-09-18 04:00:00+00:00  337.960  338.410  332.545  334.775   
GOOGL  2026-09-17 04:00:00+00:00  348.630  349.450  343.920  347.410   
       2026-09-18 04:00:00+00:00  357.305  359.365  348.910  349.520   
JPM    2026-09-17 04:00:00+00:00  352.245  353.405  345.335  349.390   
       2026-09-18 04:00:00+00:00  346.770  348.710  345.200  346.830   
NVDA   2026-09-17 04:00:00+00:00  218.180  219.900  217.145  219.400   
       2026-09-18 04:00:00+00:00  219.065  220.880  218.040  220.025   
AMZN   2026-09-17 04:00:00+00:00  250.830  252.710  249.280  251.120   
       2026-09-18 04:00:00+00:00  252.865  255.400  251.885  253.110   
JNJ    2026-09-17 04:00:00+00:00  268.170  270.850  267.425  270.255   
       2026-09-18 04:00:00+00:00  268.745  270.010  267.270  269.690   
META   2026-09-17 04:00:00+00:00  680.190  683.200  667.130  682.390   
       2026-09-18 04:00:00+00:00  687.770  690.000  668.680  671.760   
MSFT   2026-09-17 04:00:00+00:00  496.835  501.400  493.275  497.670   
       2026-09-18 04:00:00+00:00  497.635  498.140  491.130  494.375   
WMT    2026-09-17 04:00:00+00:00  107.570  107.570  106.155  106.770   
       2026-09-18 04:00:00+00:00  106.515  107.710  106.515  107.180   
XOM    2026-09-17 04:00:00+00:00  162.020  163.385  161.465  163.250   
       2026-09-18 04:00:00+00:00  162.460  163.365  162.255  162.550   

                                     volume  trade_count        vwap  
symbol timestamp                                                      
AAPL   2026-09-17 04:00:00+00:00   894177.0      21262.0  334.916974  
       2026-09-18 04:00:00+00:00   692482.0      14184.0  334.847499  
GOOGL  2026-09-17 04:00:00+00:00   642188.0      15819.0  346.224609  
       2026-09-18 04:00:00+00:00   757712.0      16670.0  353.199689  
JPM    2026-09-17 04:00:00+00:00   282564.0       8974.0  348.862665  
       2026-09-18 04:00:00+00:00   120389.0       3599.0  346.891123  
NVDA   2026-09-17 04:00:00+00:00  2712846.0      37135.0  219.051029  
       2026-09-18 04:00:00+00:00  1540956.0      18356.0  219.508126  
AMZN   2026-09-17 04:00:00+00:00   972567.0      14067.0  251.271326  
       2026-09-18 04:00:00+00:00   754519.0       9537.0  253.081432  
JNJ    2026-09-17 04:00:00+00:00   232169.0       3745.0  268.931526  
       2026-09-18 04:00:00+00:00    81068.0       1975.0  268.670441  
META   2026-09-17 04:00:00+00:00   489698.0      13409.0  676.163042  
       2026-09-18 04:00:00+00:00   369858.0       7731.0  674.841695  
MSFT   2026-09-17 04:00:00+00:00   534473.0      12452.0  496.589514  
       2026-09-18 04:00:00+00:00   363531.0       8550.0  493.594158  
WMT    2026-09-17 04:00:00+00:00   723457.0       8453.0  106.649507  
       2026-09-18 04:00:00+00:00   512574.0       5292.0  107.344801  
XOM    2026-09-17 04:00:00+00:00   351790.0       5903.0  162.457417  
       2026-09-18 04:00:00+00:00   145281.0       2360.0  162.894522

## 5. Detecting Large-Drop Signals

The V1 trading signal is defined as a close-to-close daily return of -8% or lower.

The signal calculation is:

$$ \text{Daily Return} = \frac{\text{Latest Close}}{\text{Previous Close}} - 1 $$

A stock qualifies when:

$$ \text{Daily Return} \leq -8\% $$

Only the two most recent completed daily bars are used for the signal calculation.

In [15]:
# Calculate the most recent close-to-close return for each stock in the V1 universe.

signal_rows = []

for ticker in TICKERS:

    # Skip a ticker cleanly if Alpaca/IEX did not return market data for it.
    if ticker not in returned_tickers:
        print(f"{ticker}: skipped because no market data was returned.")
        continue

    # Isolate this stock's daily bars from the MultiIndex DataFrame.
    ticker_data = market_data.loc[ticker].copy()
    ticker_data = ticker_data.sort_index()

    # Ignore today's still-forming daily bar while the market is open.
    # The signal must compare two completed trading sessions.
    if clock.is_open and len(ticker_data) > 0:
        latest_bar_date = ticker_data.index[-1].date()
        current_market_date = clock.timestamp.date()

        if latest_bar_date == current_market_date:
            ticker_data = ticker_data.iloc[:-1]

    # Two completed daily bars are required for a close-to-close return.
    if len(ticker_data) < 2:
        print(f"{ticker}: insufficient completed daily bars for signal calculation.")
        continue

    previous_bar = ticker_data.iloc[-2]
    latest_bar = ticker_data.iloc[-1]

    previous_close = float(previous_bar["close"])
    latest_close = float(latest_bar["close"])

    daily_return = (latest_close / previous_close) - 1

    signal_rows.append({
        "ticker": ticker,
        "previous_close": previous_close,
        "latest_close": latest_close,
        "daily_return": daily_return
    })


In [16]:
import pandas as pd

signal_table = pd.DataFrame(signal_rows)

signal_table

,ticker,previous_close,latest_close,daily_return
0,AAPL,332.49,337.090,0.013835
1,MSFT,490.45,497.670,0.014721
2,AMZN,245.99,251.120,0.020855
3,GOOGL,342.94,347.410,0.013034
4,META,673.59,682.390,0.013064
5,JPM,349.02,349.390,0.001060
6,JNJ,267.30,270.255,0.011055
7,XOM,163.36,163.250,-0.000673
8,WMT,107.51,106.770,-0.006883
9,NVDA,213.94,219.400,0.025521


In [17]:
# Apply the V1 large-drop rule.

#add a new col to signal_table called signal
signal_table["signal"] = (
    signal_table["daily_return"] <= DROP_THRESHOLD
)

signal_table

,ticker,previous_close,latest_close,daily_return,signal
0,AAPL,332.49,337.090,0.013835,False
1,MSFT,490.45,497.670,0.014721,False
2,AMZN,245.99,251.120,0.020855,False
3,GOOGL,342.94,347.410,0.013034,False
4,META,673.59,682.390,0.013064,False
5,JPM,349.02,349.390,0.001060,False
6,JNJ,267.30,270.255,0.011055,False
7,XOM,163.36,163.250,-0.000673,False
8,WMT,107.51,106.770,-0.006883,False
9,NVDA,213.94,219.400,0.025521,False


In [18]:
# Create a readable version for inspection.

signal_display = signal_table.copy()

signal_display["daily_return"] = (
    signal_display["daily_return"] * 100
).round(2)

signal_display = signal_display.sort_values(
    "daily_return"
)

signal_display

,ticker,previous_close,latest_close,daily_return,signal
8,WMT,107.51,106.770,-0.69,False
7,XOM,163.36,163.250,-0.07,False
5,JPM,349.02,349.390,0.11,False
6,JNJ,267.30,270.255,1.11,False
3,GOOGL,342.94,347.410,1.30,False
4,META,673.59,682.390,1.31,False
0,AAPL,332.49,337.090,1.38,False
1,MSFT,490.45,497.670,1.47,False
2,AMZN,245.99,251.120,2.09,False
9,NVDA,213.94,219.400,2.55,False


In [19]:
# Extract only stocks that satisfy the V1 trading rule.

trade_candidates = signal_table[
    signal_table["signal"]
].copy()

print(f"Qualifying signals detected: {len(trade_candidates)}")
print()

if len(trade_candidates) == 0:
    print("No V1 trade signals detected.")
else:
    for _, row in trade_candidates.iterrows():
        print(
            f"{row['ticker']}: "
            f"{row['daily_return']:.2%} daily return"
        )

Qualifying signals detected: 0

No V1 trade signals detected.


In [20]:
# Confirm that every candidate genuinely satisfies
# the configured V1 threshold.

if not trade_candidates.empty:
    
    if not (
        trade_candidates["daily_return"] <= DROP_THRESHOLD
    ).all():
        
        raise RuntimeError(
            "Signal validation failed: "
            "a candidate does not satisfy the V1 threshold."
        )

print("Signal validation complete.")

Signal validation complete.


## 6. Checking Existing Positions and Orders

Before the bot is allowed to submit any paper orders, it must check whether a qualifying stock is already being held or already has an active order.

This prevents accidental duplicate entries if the notebook is run more than once during the same trading session.

For each potential trade candidate, the bot will check:

- Whether an open position already exists for the stock.
- Whether an open order already exists for the stock.
- Whether the candidate is therefore safe to trade.

This section does not submit any new orders. It only inspects the current paper account state and removes duplicate or conflicting candidates.

In [21]:
# Retrieve all currently open paper positions.

open_positions = trading_client.get_all_positions()

print(f"Open positions: {len(open_positions)}")

for position in open_positions:
    print(
        f"{position.symbol}: "
        f"{position.qty} shares, "
        f"market value ${float(position.market_value):,.2f}"
    )

Open positions: 0


In [22]:
from alpaca.trading.requests import GetOrdersRequest
from alpaca.trading.enums import QueryOrderStatus

# Retrieve all currently open paper orders.

open_orders_request = GetOrdersRequest(
    status=QueryOrderStatus.OPEN
)

open_orders = trading_client.get_orders(
    filter=open_orders_request
)

print(f"Open orders: {len(open_orders)}")

for order in open_orders:
    print(
        f"{order.symbol}: "
        f"{order.side} "
        f"{order.qty if order.qty is not None else order.notional}"
    )

Open orders: 0


In [23]:
# Convert the current positions and orders into simple
# ticker sets for easier duplicate checking.

position_tickers = {
    position.symbol
    for position in open_positions
}

open_order_tickers = {
    order.symbol
    for order in open_orders
}

print("Current position tickers:", sorted(position_tickers))
print("Current open-order tickers:", sorted(open_order_tickers))

Current position tickers: []
Current open-order tickers: []


In [24]:
# Remove any candidate that already has either:
# 1. an open position, or
# 2. an open order.

safe_trade_candidates = []

for _, row in trade_candidates.iterrows():
    
    ticker = row["ticker"]
    
    already_positioned = ticker in position_tickers
    already_ordered = ticker in open_order_tickers
    
    if already_positioned:
        print(
            f"{ticker} skipped: "
            "an open position already exists."
        )
        continue
    
    if already_ordered:
        print(
            f"{ticker} skipped: "
            "an open order already exists."
        )
        continue
    
    safe_trade_candidates.append(row)

safe_trade_candidates = pd.DataFrame(safe_trade_candidates)

In [25]:
# Create a filtered DataFrame while preserving
# the original candidate-table structure.

blocked_tickers = position_tickers.union(open_order_tickers)

safe_trade_candidates = trade_candidates[
    ~trade_candidates["ticker"].isin(blocked_tickers)
].copy()

print(f"Original trade candidates: {len(trade_candidates)}")
print(f"Safe trade candidates: {len(safe_trade_candidates)}")

Original trade candidates: 0
Safe trade candidates: 0


In [26]:
print(f"Open positions: {len(open_positions)}")
print(f"Open orders: {len(open_orders)}")
print(f"Original trade candidates: {len(trade_candidates)}")
print(f"Safe trade candidates: {len(safe_trade_candidates)}")

if len(safe_trade_candidates) == 0:
    print("No new orders are currently required.")
else:
    print()
    print("Safe candidates:")
    
    for _, row in safe_trade_candidates.iterrows():
        print(
            f"{row['ticker']}: "
            f"{row['daily_return']:.2%}"
        )

Open positions: 0
Open orders: 0
Original trade candidates: 0
Safe trade candidates: 0
No new orders are currently required.


## 7. Position Sizing

Once a valid trading signal has been identified and confirmed to be safe for execution, the bot must determine the size of the position.

V1 uses a simple fixed-fraction allocation approach:

- Each qualifying signal receives **5% of current account equity**.
- Position size is calculated using **current account equity**.
- The resulting dollar allocation is converted into an approximate **share quantity** using the latest available stock price.

In [27]:
# Retrieve current account equity.

account = trading_client.get_account()

account_equity = float(account.equity)

print(f"Current account equity: ${account_equity:,.2f}")

# Calculate V1 position size.
position_dollars = account_equity * POSITION_SIZE_PCT

print(
    f"Position allocation per signal: "
    f"${position_dollars:,.2f}"
)

Current account equity: $100,000.00
Position allocation per signal: $5,000.00


In [28]:
# Calculate a position size for each safe trade candidate.

position_plan = []

for _, row in safe_trade_candidates.iterrows():
    
    ticker = row["ticker"]

    # Use the latest close as an approximate sizing price.
    # Actual execution will occur later.
    reference_price = float(row["latest_close"])

    shares = int(
        position_dollars // reference_price
    )

    notional_value = shares * reference_price

    position_plan.append({
        "ticker": ticker,
        "reference_price": reference_price,
        "shares": shares,
        "notional_value": notional_value
    })

position_plan = pd.DataFrame(position_plan)

position_plan

""


In [29]:
# Verify that every planned trade has a valid share count.

if not position_plan.empty:

    invalid_positions = (
        position_plan["shares"] <= 0
    ).sum()

    if invalid_positions > 0:
        raise RuntimeError(
            "One or more positions have zero shares."
        )

print("Position sizing validation complete.")

Position sizing validation complete.


In [30]:
print("V1 Trade Plan")
print("-" * 40)

if position_plan.empty:
    print("No positions required.")
else:

    total_capital = 0

    for _, row in position_plan.iterrows():

        total_capital += row["notional_value"]

        print(
            f"{row['ticker']}: "
            f"{int(row['shares'])} shares "
            f"@ ~${row['reference_price']:.2f} "
            f"(~${row['notional_value']:,.2f})"
        )

    print()
    print(
        f"Total planned capital: "
        f"${total_capital:,.2f}"
    )

V1 Trade Plan
----------------------------------------
No positions required.


## 8. Submit Buy Orders

Once a qualifying signal has passed the duplicate-order checks and a position size has been calculated, the bot can submit a **simulated buy order** to the paper-trading account.

For each planned position, the bot will:

1. Confirm that the US equity market is currently open.
2. Confirm that the calculated share quantity is greater than zero.
3. Submit a paper market buy order.
4. Attach a strategy-specific client order ID so the trade can be identified later.
5. Store the returned order information for fill tracking in the next section.

In [31]:
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

print("Order submission tools loaded.")

Order submission tools loaded.


In [32]:
# Build the paper buy orders from the position plan.

planned_buy_orders = []

# Use the broker's current market date in the client order ID so
# the same ticker can trade again on a different session.
order_session_date = trading_client.get_clock().timestamp.strftime("%Y%m%d")

for _, row in position_plan.iterrows():

    ticker = row["ticker"]
    shares = int(row["shares"])

    # Safety check:
    # Never construct an order with zero or negative shares.
    if shares <= 0:
        print(
            f"{ticker} skipped: "
            "calculated share quantity is not valid."
        )
        continue

    # Create a strategy-specific order ID that is unique across trading days.
    client_order_id = (
        f"{STRATEGY_NAME}_{ticker}_buy_{order_session_date}"
    )

    order_request = MarketOrderRequest(
        symbol=ticker,
        qty=shares,
        side=OrderSide.BUY,
        time_in_force=TimeInForce.DAY, #order is valid for the current trading day only
        client_order_id=client_order_id
    )

    planned_buy_orders.append({
        "ticker": ticker,
        "shares": shares,
        "client_order_id": client_order_id,
        "order_request": order_request
    })

print(
    f"Paper buy orders prepared: "
    f"{len(planned_buy_orders)}"
)


Paper buy orders prepared: 0


In [33]:
#inspect prepared buy orders before sending to paper-trading environment 
print("Prepared V1 Buy Orders")
print("-" * 40)

if len(planned_buy_orders) == 0:
    print("No buy orders are currently required.")

else:

    for order in planned_buy_orders:

        print(
            f"{order['ticker']}: "
            f"BUY {order['shares']} shares "
            f"| ID: {order['client_order_id']}"
        )

Prepared V1 Buy Orders
----------------------------------------
No buy orders are currently required.


In [34]:
# Submit prepared orders only when all execution safety conditions pass.
submitted_orders = []

print("Execution Safety Check")
print("-" * 40)

# Safety condition 1:
# This version of the bot must only operate in paper-trading mode.
if PAPER_TRADING is not True:
    raise RuntimeError(
        "Execution blocked: PAPER_TRADING must be explicitly set to True."
    )

print("✓ Paper trading mode confirmed.")

# Safety condition 2:
# Only submit new market orders while the market is open.
if not clock.is_open:
    print("✗ Market is currently closed. No orders will be submitted.")

# Safety condition 3:
# There must actually be qualifying orders to send.
elif len(planned_buy_orders) == 0:
    print("✓ Market is open.")
    print("✗ No qualifying buy orders to submit.")

else:
    print("✓ Market is open.")
    print(
        f"Submitting {len(planned_buy_orders)} "
        "paper buy order(s)..."
    )

    for order in planned_buy_orders:

        try:
            submitted_order = trading_client.submit_order(
                order_data=order["order_request"]
            )

            submitted_orders.append({
                "ticker": order["ticker"],
                "shares": order["shares"],
                "client_order_id": order["client_order_id"],
                "alpaca_order_id": str(submitted_order.id),
                "status": str(submitted_order.status)
            })

            print(
                f"✓ {order['ticker']}: "
                f"submitted BUY {order['shares']} shares "
                f"| status = {submitted_order.status}"
            )

        except Exception as e:

            print(
                f"✗ {order['ticker']}: "
                f"order submission failed | {e}"
            )

print("-" * 40)
print(f"Orders successfully submitted: {len(submitted_orders)}")

Execution Safety Check
----------------------------------------
✓ Paper trading mode confirmed.
✓ Market is open.
✗ No qualifying buy orders to submit.
----------------------------------------
Orders successfully submitted: 0


## 9. Track Fills

Submitting an order does not guarantee that the trade has executed.

After sending a buy order to Alpaca, the bot checks the broker's order record to determine whether the order was filled, is still pending, or failed to execute.

For each submitted order, this section records:

- the Alpaca order ID,
- the current order status,
- the requested quantity,
- the filled quantity,
- the average fill price, and
- the fill timestamp.

This creates a clear distinction between an **order request** and an **executed position**. Only confirmed fills should be used by later sections of the strategy.

In [35]:
# Retrieve the latest broker status for each order submitted by V1.

tracked_orders = []

print("Order Fill Tracking")
print("-" * 40)

if len(submitted_orders) == 0:
    print("No submitted orders to track.")

else:

    for order in submitted_orders:

        try:
            broker_order = trading_client.get_order_by_id(
                order["alpaca_order_id"]
            )

            tracked_orders.append({
                "ticker": order["ticker"],
                "alpaca_order_id": str(broker_order.id), #unique ID to track the order from previous section
                "client_order_id": broker_order.client_order_id,
                "status": str(broker_order.status),
                "requested_qty": float(broker_order.qty), #How many shares did we ask for?
                "filled_qty": float(broker_order.filled_qty), #How many shares actually executed?
                #the price at which the stock is actually bought / order is executed (can differ from reference price)
                "filled_avg_price": (
                    float(broker_order.filled_avg_price)
                    if broker_order.filled_avg_price is not None
                    else None
                ),
                "filled_at": broker_order.filled_at
            })

            print(
                f"{order['ticker']}: "
                f"status = {broker_order.status} "
                f"| filled = {broker_order.filled_qty}/"
                f"{broker_order.qty}"
            )

        except Exception as e:

            print(
                f"✗ {order['ticker']}: "
                f"could not retrieve order status | {e}"
            )

print("-" * 40)
print(f"Orders tracked: {len(tracked_orders)}")

Order Fill Tracking
----------------------------------------
No submitted orders to track.
----------------------------------------
Orders tracked: 0


In [36]:
#create a dataframe to track this information
tracked_orders_df = pd.DataFrame(tracked_orders)

if tracked_orders_df.empty:
    print("No fill records available.")

else:
    display(
        tracked_orders_df[
            [
                "ticker", #stock name
                "status", #filled, not filled, partially filled by broker 
                "requested_qty", #quantity of stock requested 
                "filled_qty", #quantity of stock bought 
                "filled_avg_price", #price at which stock was bought
                "filled_at" #timestamp at which stock was bought
            ]
        ]
    )

No fill records available.


## 10. Track Open Positions

**Orders** describe instructions sent to the broker, while **positions** describe assets that are held in the account.

After checking order fills, the bot retrieves all open positions from the Alpaca paper account and identifies positions associated with the V1 trading universe.

For each open position, the bot records:

- ticker,
- quantity held,
- average entry price,
- current market price,
- market value,
- unrealized profit or loss, and
- unrealized return.

This provides a current snapshot of the portfolio before the strategy evaluates whether any positions need to be closed.

In [37]:
# Retrieve all currently open positions from the paper account.

v1_open_positions = []

print("Open Position Tracking")
print("-" * 40)

try:
    broker_positions = trading_client.get_all_positions()

    for position in broker_positions:

        # Only include stocks that belong to the V1 strategy universe.
        if position.symbol not in TICKERS:
            continue

        v1_open_positions.append({
            "ticker": position.symbol,
            "qty": float(position.qty),
            "avg_entry_price": float(position.avg_entry_price), #price at which share was acquired
            "current_price": float(position.current_price), #current price of the stock
            "market_value": float(position.market_value), #share x current price
            "unrealized_pl": float(position.unrealized_pl), #dollar P&L e.g. +$63.20
            "unrealized_plpc": float(position.unrealized_plpc) #proportional return 0.0124 --> +1.24%
        })

except Exception as e:
    print(f"✗ Could not retrieve open positions | {e}")

print(f"V1 universe positions currently open: {len(v1_open_positions)}")

Open Position Tracking
----------------------------------------
V1 universe positions currently open: 0


In [38]:
v1_open_positions_df = pd.DataFrame(v1_open_positions)

if v1_open_positions_df.empty:
    print("No V1 universe positions are currently open.")

else:
    display_positions = v1_open_positions_df.copy()

    display_positions["unrealized_return_pct"] = (
        display_positions["unrealized_plpc"] * 100
    )

    display(
        display_positions[
            [
                "ticker",
                "qty",
                "avg_entry_price",
                "current_price",
                "market_value",
                "unrealized_pl",
                "unrealized_return_pct"
            ]
        ].round(2)
    )

No V1 universe positions are currently open.


## 11. Submit Sell Orders Near Close

The V1 strategy determines that a stock is bought following an extreme decline and sold near the end of that same trading session (day).  

The bot only considers positions for exit when:

1. the position was created by a V1 buy order submitted during the current session,
2. that buy order has been confirmed as filled,
3. the position is still open, and
4. the market has reached the designated exit window near the closing bell.

V1 uses a configurable exit window instead of attempting to submit an order at the exact closing second. This reduces the risk of missing the session.

In [39]:
# Refresh the market clock before making an exit decision.

clock = trading_client.get_clock()

seconds_until_close = (
    clock.next_close - clock.timestamp
).total_seconds()

minutes_until_close = seconds_until_close / 60

print("V1 Exit Timing Check")
print("-" * 40)
print(f"Market open: {clock.is_open}")

if clock.is_open:
    print(
        f"Minutes until market close: "
        f"{minutes_until_close:.1f}"
    )
else:
    print("Market is currently closed.")

V1 Exit Timing Check
----------------------------------------
Market open: True
Minutes until market close: 161.4


In [40]:
# Determine whether V1 is currently inside its exit window.

exit_window_active = (
    clock.is_open
    and 0 <= minutes_until_close <= EXIT_MINUTES_BEFORE_CLOSE
)

if exit_window_active:
    print(
        f"✓ V1 exit window is active "
        f"(≤ {EXIT_MINUTES_BEFORE_CLOSE} minutes to close)."
    )
else:
    print(
        f"V1 exit window is not active YET..."
        f"Target exit window is {EXIT_MINUTES_BEFORE_CLOSE} minutes before market close."
    )

V1 exit window is not active YET...Target exit window is 10 minutes before market close.


In [41]:
# Refresh V1 entry-order statuses before making the exit decision.
# An order that was still pending when Section 9 first checked it may
# have filled later in the session.

refreshed_tracked_orders = []

for order in tracked_orders:

    try:
        broker_order = trading_client.get_order_by_id(
            order["alpaca_order_id"]
        )

        refreshed_tracked_orders.append({
            "ticker": order["ticker"],
            "alpaca_order_id": str(broker_order.id),
            "client_order_id": broker_order.client_order_id,
            "status": str(broker_order.status),
            "requested_qty": float(broker_order.qty),
            "filled_qty": float(broker_order.filled_qty),
            "filled_avg_price": (
                float(broker_order.filled_avg_price)
                if broker_order.filled_avg_price is not None
                else None
            ),
            "filled_at": broker_order.filled_at
        })

    except Exception as e:
        print(
            f"✗ {order['ticker']}: "
            f"could not refresh entry order status | {e}"
        )

# Replace the earlier snapshot with the latest broker state.
tracked_orders = refreshed_tracked_orders

# Identify fully filled V1 entry orders from this notebook run.
v1_filled_entries = {}

for order in tracked_orders:

    fully_filled = (
        float(order["filled_qty"]) > 0
        and float(order["filled_qty"]) == float(order["requested_qty"])
        and order["filled_avg_price"] is not None
    )

    if fully_filled:
        v1_filled_entries[order["ticker"]] = order

print(f"Fully filled V1 entries identified: {len(v1_filled_entries)}")

# Match today's fully filled V1 entries to positions still open at the broker.
exit_candidates = []

for position in v1_open_positions:

    ticker = position["ticker"]

    if ticker not in v1_filled_entries:
        continue

    entry = v1_filled_entries[ticker]

    # Only manage the quantity attributable to this V1 entry.
    v1_entry_qty = float(entry["filled_qty"])
    broker_position_qty = float(position["qty"])

    exit_qty = min(v1_entry_qty, broker_position_qty)

    if exit_qty <= 0:
        continue

    exit_candidates.append({
        "ticker": ticker,
        "qty": exit_qty,
        "avg_entry_price": float(entry["filled_avg_price"]),
        "current_price": position["current_price"],
        "entry_order_id": entry["alpaca_order_id"]
    })

print(f"Confirmed V1 positions eligible for exit management: {len(exit_candidates)}")


Fully filled V1 entries identified: 0
Confirmed V1 positions eligible for exit management: 0


In [42]:
# Submit same-session paper exit orders when the exit window is active.

submitted_exit_orders = []

print("V1 Exit Execution")
print("-" * 40)

if PAPER_TRADING is not True:
    raise RuntimeError(
        "Exit execution blocked: PAPER_TRADING must be explicitly True."
    )

print("✓ Paper trading mode confirmed.")

if not clock.is_open:
    print("✗ Market is closed. Exit orders cannot be submitted.")

elif not exit_window_active:
    print("✗ Exit window is not currently active. No positions will be closed.")

elif len(exit_candidates) == 0:
    print("No confirmed V1 positions require an exit.")

else:

    print(
        f"Submitting {len(exit_candidates)} "
        "paper exit order(s)..."
    )

    # Use the refreshed broker date for strategy-specific sell order IDs.
    exit_session_date = clock.timestamp.strftime("%Y%m%d")

    for position in exit_candidates:

        ticker = position["ticker"]
        qty = position["qty"]

        # Convert whole-share quantities back to an integer where possible.
        if float(qty).is_integer():
            qty = int(qty)

        client_order_id = (
            f"{STRATEGY_NAME}_{ticker}_sell_{exit_session_date}"
        )

        exit_request = MarketOrderRequest(
            symbol=ticker,
            qty=qty,
            side=OrderSide.SELL,
            time_in_force=TimeInForce.DAY,
            client_order_id=client_order_id
        )

        try:
            submitted_exit = trading_client.submit_order(
                order_data=exit_request
            )

            submitted_exit_orders.append({
                "ticker": ticker,
                "qty": qty,
                "client_order_id": client_order_id,
                "alpaca_order_id": str(submitted_exit.id),
                "status": str(submitted_exit.status)
            })

            print(
                f"✓ {ticker}: "
                f"submitted SELL {qty} shares "
                f"| status = {submitted_exit.status}"
            )

        except Exception as e:

            print(
                f"✗ {ticker}: "
                f"exit submission failed | {e}"
            )

print("-" * 40)
print(
    f"Exit orders successfully submitted: "
    f"{len(submitted_exit_orders)}"
)


V1 Exit Execution
----------------------------------------
✓ Paper trading mode confirmed.
✗ Exit window is not currently active. No positions will be closed.
----------------------------------------
Exit orders successfully submitted: 0


## 12. Record Results

A trading system needs a persistent record of what it has actually done.

After V1 submits an exit order, this section checks the broker for the final execution details and records completed trades in a local CSV file. Each trade record connects the original entry with its corresponding exit and stores:

- ticker,
- entry and exit timestamps,
- quantity,
- entry and exit prices,
- gross return,
- estimated net return,
- profit or loss, and
- broker order IDs.

The log is stored outside the notebook so results remain available between sessions. This will allow future paper trades to be evaluated as genuine out-of-sample observations rather than disappearing when the notebook is restarted.

In [43]:
from pathlib import Path

TRADE_LOG_PATH = Path("v1_trade_log.csv")

TRADE_LOG_COLUMNS = [
    "strategy",
    "ticker",
    "entry_time",
    "exit_time",
    "qty",
    "entry_price",
    "exit_price",
    "gross_return_pct",
    "estimated_net_return_pct",
    "pnl_dollars",
    "entry_order_id",
    "exit_order_id"
]

print(f"Trade log location: {TRADE_LOG_PATH.resolve()}")

Trade log location: C:\Users\amina\stock-bot\v1_trade_log.csv


In [44]:
# Load the existing trade history, or create an empty log.

if TRADE_LOG_PATH.exists():

    trade_log_df = pd.read_csv(TRADE_LOG_PATH)

    print(
        f"Existing trade log loaded: "
        f"{len(trade_log_df)} completed trade(s)."
    )

else:

    trade_log_df = pd.DataFrame(
        columns=TRADE_LOG_COLUMNS
    )

    print("No existing trade log found. A new log will be created.")

No existing trade log found. A new log will be created.


In [45]:
# Match confirmed exit fills with their corresponding V1 entry fills.

completed_trades = []

for exit_order in submitted_exit_orders:
    try:
        broker_exit = trading_client.get_order_by_id(exit_order["alpaca_order_id"])

        # Record the trade only after the full requested exit quantity has filled.
        if (
            broker_exit.filled_qty is None
            or broker_exit.qty is None
            or broker_exit.filled_avg_price is None
            or float(broker_exit.filled_qty) <= 0
            or float(broker_exit.filled_qty) < float(broker_exit.qty)
        ):
            print(f"{exit_order['ticker']}: exit is not fully filled yet.")
            continue

        ticker = exit_order["ticker"]
        matching_entries = [
            order for order in tracked_orders
            if (
                order["ticker"] == ticker
                and float(order["filled_qty"]) > 0
                and float(order["filled_qty"]) == float(order["requested_qty"])
                and order["filled_avg_price"] is not None
            )
        ]

        if len(matching_entries) == 0:
            print(f"✗ {ticker}: no matching V1 entry fill found.")
            continue

        entry = matching_entries[-1]
        qty = min(float(entry["filled_qty"]), float(broker_exit.filled_qty))
        entry_price = float(entry["filled_avg_price"])
        exit_price = float(broker_exit.filled_avg_price)
        gross_return = (exit_price / entry_price) - 1
        pnl_dollars = (exit_price - entry_price) * qty

        completed_trades.append({
            "strategy": STRATEGY_NAME,
            "ticker": ticker,
            "entry_time": entry["filled_at"],
            "exit_time": broker_exit.filled_at,
            "qty": qty,
            "entry_price": entry_price,
            "exit_price": exit_price,
            "gross_return_pct": gross_return * 100,
            "estimated_net_return_pct": None,
            "pnl_dollars": pnl_dollars,
            "entry_order_id": entry["alpaca_order_id"],
            "exit_order_id": str(broker_exit.id)
        })

    except Exception as e:
        print(f"✗ {exit_order['ticker']}: could not build completed trade record | {e}")

print(f"New completed trades found: {len(completed_trades)}")


New completed trades found: 0


In [46]:
# Append newly completed trades while preventing duplicate records.

if len(completed_trades) > 0:

    new_trades_df = pd.DataFrame(completed_trades)

    if not trade_log_df.empty:

        existing_exit_ids = set(
            trade_log_df["exit_order_id"].astype(str)
        )

        new_trades_df = new_trades_df[
            ~new_trades_df["exit_order_id"]
            .astype(str)
            .isin(existing_exit_ids)
        ]

    if not new_trades_df.empty:

        trade_log_df = pd.concat(
            [trade_log_df, new_trades_df],
            ignore_index=True
        )

        trade_log_df.to_csv(
            TRADE_LOG_PATH,
            index=False
        )

        print(
            f"✓ Saved {len(new_trades_df)} "
            "new completed trade(s)."
        )

    else:
        print("No new unique trades to save.")

else:
    print("No completed trades to record.")

No completed trades to record.


In [47]:
if trade_log_df.empty:

    print("Trade log is currently empty.")

else:

    display_log = trade_log_df.copy()

    display(
        display_log[
            [
                "ticker",
                "entry_time",
                "exit_time",
                "qty",
                "entry_price",
                "exit_price",
                "gross_return_pct",
                "pnl_dollars"
            ]
        ].round(2)
    )

Trade log is currently empty.


## 13. Print Daily Summary
A simple end-of-run audit of what the bot observed and what actions it took.

In [48]:
# Refresh the paper account for the final session summary.

account = trading_client.get_account()

confirmed_fills = sum(
    1 for order in tracked_orders
    if (
        float(order["filled_qty"]) > 0
        and float(order["filled_qty"]) == float(order["requested_qty"])
        and order["filled_avg_price"] is not None
    )
)

print("=" * 55)
print("STOCK BOT V1 — DAILY SUMMARY")
print("=" * 55)
print(f"Strategy:              {STRATEGY_NAME}")
print(f"Paper trading:         {PAPER_TRADING}")
print(f"Stocks scanned:        {len(TICKERS)}")
print(f"Signals detected:      {len(trade_candidates)}")
print(f"Eligible entries:      {len(safe_trade_candidates)}")
print(f"Buy orders submitted:  {len(submitted_orders)}")
print(f"Confirmed buy fills:   {confirmed_fills}")
print(f"Managed V1 positions:  {len(exit_candidates)}")
print(f"Exit orders submitted: {len(submitted_exit_orders)}")
print(f"Trades completed:      {len(completed_trades)}")
print("-" * 55)
print(f"Account equity:        ${float(account.equity):,.2f}")
print(f"Cash:                  ${float(account.cash):,.2f}")
print("-" * 55)

if len(trade_candidates) == 0:
    print("Session result: No qualifying extreme-drop signals.")
elif len(safe_trade_candidates) == 0:
    print("Session result: Signals detected, but no new entries were eligible.")
elif len(submitted_orders) == 0:
    print("Session result: Eligible signals found, but no buy orders were submitted.")
elif len(completed_trades) > 0:
    print(f"Session result: {len(completed_trades)} trade(s) completed.")
else:
    print("Session result: V1 activity detected; trade lifecycle remains in progress.")

print("=" * 55)


STOCK BOT V1 — DAILY SUMMARY
Strategy:              large_drop_v1
Paper trading:         True
Stocks scanned:        10
Signals detected:      0
Eligible entries:      0
Buy orders submitted:  0
Confirmed buy fills:   0
Managed V1 positions:  0
Exit orders submitted: 0
Trades completed:      0
-------------------------------------------------------
Account equity:        $100,000.00
Cash:                  $100,000.00
-------------------------------------------------------
Session result: No qualifying extreme-drop signals.


### V1 execution limitation

This notebook manages entry and exit state in memory. As a result, the current V1 implementation assumes that the same notebook session remains active between entry execution and the end-of-session exit.

If the notebook kernel is restarted after an entry has been filled, the in-memory `tracked_orders` object is lost and V1 will not automatically reconstruct strategy ownership from the broker account.

A future version will persist open strategy state independently of the notebook session and use scheduled execution for entry and exit management.